# Week 1: statistical baselines
Based on the corrected `Week1_code.ipynb` baseline analysis. The separate starter notebook is supplied tutorial code.

Compare SK, bandpass and their combined flagger on all **60 validation recordings × 10 antennas**, using native-resolution masks. Sum TP/FP/FN/TN before computing precision, recall or FPR. Single examples illustrate errors; they do not replace the pooled results.

In [ ]:
from cs163_rfi import paths, load_sample, flag_rfi, score_mask
from cs163_rfi.baselines import evaluate_validation
from cs163_rfi.plotting import plot_sample, plot_errors
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display
OUT = paths.RESULTS_DIR / "week_01"
METHODS = ("sk", "bandpass", "sk-bandpass")
FAR = 1e-4

## Pooled validation comparison
SK tests power fluctuations at accumulation lengths 64, 256, 1024 and 4096. Bandpass tests channel-power outliers. The combined method takes their union and expands frequency clusters.

Nominal FAR controls internal thresholds; it is not measured FPR. Flagged channels cover a full 4096-sample block. Labels are positive when injected RFI power reaches 1% of mean receiver-noise power; clean and sky controls have no RFI labels. Detectors receive unnormalized power.

In [ ]:
RUN_EVALUATION = False  # Set True to recompute all validation masks.
if RUN_EVALUATION:
    summary = evaluate_validation(far=FAR)
else:
    summary = pd.read_csv(OUT / "baseline_validation.csv", index_col="method")
display(summary[["tp", "fp", "fn", "tn", "precision", "recall", "false_positive_rate"]])

## Example masks
First inspect training recording 18, antenna 0. Then select a missed-RFI case and a high-FPR case from the validation counts. Plots may average pixels for display; scores always use native masks.

In [ ]:
sample = load_sample(paths.DATA_DIR, 18, antenna=0)
for method in METHODS:
    flags = flag_rfi(sample["power"], method=method, far=FAR)
    print(method, score_mask(flags, sample["rfi_mask"]))
    plot_sample(sample, flags, factor=16)
    plt.show()

In [ ]:
counts = pd.read_csv(OUT / "baseline_per_antenna.csv")
combined = counts[counts.method.eq("sk-bandpass")].copy()
combined["targets"] = combined.tp + combined.fn
combined["recall"] = combined.tp / combined.targets.replace(0, np.nan)
combined["fpr"] = combined.fp / (combined.fp + combined.tn)
miss = combined[combined.targets.gt(0)].sort_values(
    ["recall", "targets", "sample_id", "antenna"], ascending=[True, False, True, True]).iloc[0]
false_flag = combined.sort_values(["fpr", "sample_id", "antenna"], ascending=[False, True, True]).iloc[0]
for label, row in [("Missed RFI", miss), ("High false-positive rate", false_flag)]:
    sample = load_sample(paths.DATA_DIR, int(row.sample_id), antenna=int(row.antenna))
    flags = flag_rfi(sample["power"], method="sk-bandpass", far=FAR)
    print(label, int(row.sample_id), int(row.antenna), score_mask(flags, sample["rfi_mask"]))
    error = sample["rfi_mask"] & ~flags if label == "Missed RFI" else flags & ~sample["rfi_mask"]
    center = int(np.argmax(error.sum(axis=1)))
    plot_errors(sample, flags, first=int(np.clip(center - 256, 0, len(flags) - 512)))
    plt.show()

## Optional FAR sweep
Compare ROC and precision–recall operating points on the same validation set. These are five detector settings, not probability-score AP curves.

In [ ]:
RUN_FAR_SWEEP = False
if RUN_FAR_SWEEP:
    sweep = []
    for far in (1e-6, 1e-5, 1e-4, 1e-3, 1e-2):
        table = evaluate_validation(far=far, output_dir=OUT / f"far_{far:g}")
        sweep.append(table.reset_index().assign(far=far))
    sweep = pd.concat(sweep, ignore_index=True)
    sweep.to_csv(OUT / "far_sweep.csv", index=False)
    fig, axes = plt.subplots(1, 2, figsize=(10, 4), layout="constrained")
    for method, rows in sweep.groupby("method"):
        axes[0].plot(rows.false_positive_rate, rows.recall, "o-", label=method)
        axes[1].plot(rows.recall, rows.precision, "o-", label=method)
    axes[0].set(xlabel="False-positive rate", ylabel="Recall")
    axes[1].set(xlabel="Recall", ylabel="Precision")
    for ax in axes:
        ax.legend()
        ax.grid(alpha=.2)
    fig.savefig(OUT / "far_sweep.png", dpi=200)
    plt.show()

## Takeaway
At FAR 1e-4, bandpass has the highest pooled precision (68.39%); the combined detector has higher recall (51.32%) but FPR 7.37%. Nominal FAR does not equal measured FPR after combining tests and expanding channels. The error examples explain the trade-off between missed RFI and removing useful samples.